# 2 – Pandas: Patientendaten als Tabelle

**Pandas** ist *das* Werkzeug für tabellarische Daten in Python – so etwas wie Excel, nur programmierbar, reproduzierbar und viel mächtiger. Die zentrale Datenstruktur heißt **DataFrame**: eine Tabelle mit benannten Spalten, in der jede Zeile eine Beobachtung (z. B. eine Patientin) ist.

### Lernziele
Nach diesem Notebook können Sie
- Daten aus CSV-Dateien laden und speichern,
- sich schnell einen Überblick über einen Datensatz verschaffen,
- Spalten auswählen und Zeilen nach Bedingungen filtern,
- neue Spalten berechnen (z. B. BMI, Hypertonie ja/nein),
- mit fehlenden Werten umgehen,
- Gruppen vergleichen (z. B. Raucher vs. Nichtraucher).

### Hinweise
- Zellen der Reihe nach mit `Shift + Enter` ausführen.
- ⭐ = wird in der Vorlesung gezeigt, Rest zur Vertiefung.
- Pandas baut auf NumPy auf – vieles aus Notebook 01 kommt Ihnen bekannt vor.

## ⭐ 1. Import und ein erster DataFrame

Pandas wird per Konvention als `pd` importiert. Einen kleinen DataFrame können Sie direkt aus einem *Dictionary* erzeugen: Jeder Schlüssel wird ein Spaltenname, jede Liste der Inhalt der Spalte.

In [ ]:
import numpy as np
import pandas as pd

mini = pd.DataFrame({
    "name":   ["Müller", "Schmidt", "Yilmaz", "Weber"],
    "alter":  [67, 45, 52, 78],
    "sys_rr": [145, 122, 138, 160],
})
mini

Die fett gedruckten Zahlen links (0, 1, 2, …) sind der **Index** – die Zeilennummern. Eine einzelne Spalte ist eine **Series** (im Grunde ein NumPy-Array mit Namen):

In [ ]:
mini["alter"]

## ⭐ 2. Einen echten (simulierten) Datensatz erzeugen und laden

Die folgende Zelle erzeugt einen **simulierten** Datensatz mit 200 Patient\*innen und speichert ihn als `patienten.csv` im selben Ordner wie dieses Notebook. Die Datei brauchen wir auch in Notebook 03. Führen Sie die Zelle einfach aus.

In [ ]:
# Simulierter Datensatz: 200 Patient*innen einer internistischen Ambulanz
# (Sie müssen diesen Code nicht im Detail verstehen.)
import numpy as np
import pandas as pd

rng = np.random.default_rng(seed=2024)
n = 200

geschlecht = rng.choice(["w", "m"], size=n)
alter      = rng.integers(18, 90, size=n)
groesse    = np.where(geschlecht == "m", rng.normal(180, 7, n), rng.normal(167, 6, n)).round(0)
bmi_wahr   = rng.normal(26.5, 4.5, n).clip(16, 45)
gewicht    = (bmi_wahr * (groesse / 100) ** 2).round(1)
raucher    = rng.random(n) < 0.25
sys_rr     = (100 + 0.5 * alter + 0.9 * (bmi_wahr - 25) + 6 * raucher + rng.normal(0, 12, n)).round(0)
dia_rr     = (0.5 * sys_rr + rng.normal(15, 6, n)).round(0)
cholesterin = (165 + 0.7 * alter + rng.normal(0, 30, n)).round(0)
p_diabetes = 1 / (1 + np.exp(-(-5.5 + 0.045 * alter + 0.15 * (bmi_wahr - 25))))
diabetes   = rng.random(n) < p_diabetes

df = pd.DataFrame({
    "patient_id": [f"P{i:03d}" for i in range(1, n + 1)],
    "alter": alter,
    "geschlecht": geschlecht,
    "groesse_cm": groesse,
    "gewicht_kg": gewicht,
    "raucher": raucher,
    "sys_rr": sys_rr,
    "dia_rr": dia_rr,
    "cholesterin": cholesterin,
    "diabetes": diabetes,
})

# Ein paar fehlende Werte einbauen – wie im echten Leben
df.loc[rng.choice(n, 8, replace=False), "cholesterin"] = np.nan
df.loc[rng.choice(n, 5, replace=False), "gewicht_kg"] = np.nan

df.to_csv("patienten.csv", index=False)
print("Datei 'patienten.csv' wurde erstellt.")

In der Praxis bekommen Sie Daten meist als Datei. Eingelesen wird mit `pd.read_csv`:

In [ ]:
df = pd.read_csv("patienten.csv")

> **Tipp für deutsche Excel-Dateien:** CSV-Dateien aus einem deutschen Excel verwenden oft `;` als Trennzeichen und `,` als Dezimalzeichen. Dann: `pd.read_csv("datei.csv", sep=";", decimal=",")`. Excel-Dateien direkt: `pd.read_excel("datei.xlsx")` (benötigt das Paket `openpyxl`).

## ⭐ 3. Überblick verschaffen

Bevor Sie irgendetwas analysieren: **Schauen Sie sich die Daten an!** Diese fünf Befehle sollten Sie bei jedem neuen Datensatz ausführen.

In [ ]:
df.head()          # die ersten 5 Zeilen (df.tail() für die letzten)

In [ ]:
df.shape           # (Anzahl Zeilen, Anzahl Spalten)

In [ ]:
df.info()          # Spalten, Datentypen und Anzahl nicht-fehlender Werte

In `info()` sehen Sie schon: Bei `gewicht_kg` und `cholesterin` sind weniger als 200 Werte vorhanden – es fehlen Daten.

In [ ]:
df.describe().round(1)   # deskriptive Statistik aller numerischen Spalten

In [ ]:
df["geschlecht"].value_counts()   # Häufigkeiten einer kategorialen Variable

## ⭐ 4. Spalten auswählen

- Eine Spalte: `df["spalte"]` → ergibt eine Series
- Mehrere Spalten: `df[["spalte1", "spalte2"]]` → ergibt einen DataFrame (**doppelte** eckige Klammern!)

In [ ]:
df["alter"].mean()

In [ ]:
df[["patient_id", "sys_rr", "dia_rr"]].head()

## ⭐ 5. Zeilen filtern

Funktioniert genau wie in NumPy mit Bedingungen. Wieder gilt: Mehrere Bedingungen mit `&` (und) bzw. `|` (oder) und **Klammern** um jede Bedingung.

In [ ]:
senioren = df[df["alter"] >= 65]
print("Anzahl Patient*innen ab 65:", len(senioren))
senioren.head()

In [ ]:
# Rauchende Männer mit systolischem Blutdruck >= 150 mmHg
df[(df["geschlecht"] == "m") & (df["raucher"]) & (df["sys_rr"] >= 150)]

### `loc` und `iloc`

Für gezielteren Zugriff gibt es zwei Werkzeuge:
- `df.loc[zeilen, spalten]` – Zugriff über **Namen/Bedingungen**
- `df.iloc[zeilen, spalten]` – Zugriff über **Positionen** (Zahlen, wie bei NumPy)

In [ ]:
# Alter und Blutdruck aller Diabetiker*innen
df.loc[df["diabetes"], ["alter", "sys_rr"]].head()

In [ ]:
# Die ersten 3 Zeilen, die ersten 4 Spalten
df.iloc[:3, :4]

## ⭐ 6. Neue Spalten berechnen

Rechnen mit Spalten funktioniert elementweise wie bei NumPy-Arrays. Das Ergebnis weisen Sie einfach einem neuen Spaltennamen zu:

In [ ]:
df["bmi"] = df["gewicht_kg"] / (df["groesse_cm"] / 100) ** 2
df["bmi"] = df["bmi"].round(1)

df["hypertonie"] = (df["sys_rr"] >= 140) | (df["dia_rr"] >= 90)

df[["patient_id", "bmi", "sys_rr", "dia_rr", "hypertonie"]].head()

Mit `pd.cut` teilen Sie eine kontinuierliche Variable in **Kategorien** ein – hier nach der WHO-Klassifikation des BMI:

In [ ]:
df["bmi_kategorie"] = pd.cut(
    df["bmi"],
    bins=[0, 18.5, 25, 30, 100],
    labels=["Untergewicht", "Normalgewicht", "Übergewicht", "Adipositas"],
    right=False,   # Intervalle [18.5, 25) usw. – 25.0 zählt also schon als Übergewicht
)
df["bmi_kategorie"].value_counts()

## ⭐ 7. Fehlende Werte

Fehlende Werte erscheinen in Pandas als `NaN`. Zuerst immer prüfen, **wo** und **wie viele** fehlen:

In [ ]:
df.isna().sum()

Die gute Nachricht: Pandas-Statistiken wie `.mean()` ignorieren `NaN` automatisch. Für manche Analysen müssen Sie aber entscheiden, was mit den Lücken passiert:

| Strategie | Code | Wann sinnvoll? |
|---|---|---|
| Zeilen entfernen | `df.dropna()` | wenige fehlende Werte, zufällig verteilt |
| Nur bestimmte Spalten berücksichtigen | `df.dropna(subset=["cholesterin"])` | wenn nur einzelne Variablen relevant sind |
| Auffüllen (Imputation) | `df["x"].fillna(df["x"].median())` | mit Vorsicht – kann Ergebnisse verzerren! |

> **Achtung:** Die meisten Methoden geben einen **neuen** DataFrame zurück und verändern das Original nicht. Sie müssen das Ergebnis einer Variablen zuweisen.

In [ ]:
df_komplett = df.dropna()
print("Vorher:", len(df), "Zeilen – nachher:", len(df_komplett), "Zeilen")

In [ ]:
median_chol = df["cholesterin"].median()
df["cholesterin_imputiert"] = df["cholesterin"].fillna(median_chol)
print("Fehlende Werte nach Imputation:", df["cholesterin_imputiert"].isna().sum())

## ⭐ 8. Gruppen vergleichen mit `groupby`

`groupby` ist eines der mächtigsten Werkzeuge in Pandas. Die Logik heißt **Split – Apply – Combine**: Daten nach einer Variable aufteilen, auf jede Gruppe eine Berechnung anwenden, Ergebnisse zusammenführen.

In [ ]:
# Mittlerer systolischer Blutdruck von Rauchern vs. Nichtrauchern
df.groupby("raucher")["sys_rr"].mean().round(1)

In [ ]:
# Mehrere Kennzahlen auf einmal mit .agg()
df.groupby("geschlecht")[["alter", "bmi", "sys_rr"]].agg(["mean", "std"]).round(1)

In [ ]:
# Anteil Diabetes je BMI-Kategorie (Mittelwert von True/False = Anteil!)
df.groupby("bmi_kategorie", observed=True)["diabetes"].mean().round(2)

### Kreuztabellen (Vierfeldertafeln)

Für zwei kategoriale Variablen gibt es `pd.crosstab` – nützlich z. B. als Vorbereitung für einen Chi-Quadrat-Test:

In [ ]:
pd.crosstab(df["raucher"], df["hypertonie"], margins=True)

In [ ]:
# Mit normalize="index": Zeilenprozente
pd.crosstab(df["raucher"], df["hypertonie"], normalize="index").round(2)

## 9. Sortieren

Mit `sort_values` sortieren Sie nach einer oder mehreren Spalten:

In [ ]:
# Die 5 Patient*innen mit dem höchsten Blutdruck
df.sort_values("sys_rr", ascending=False)[["patient_id", "alter", "sys_rr"]].head(5)

## 10. Daten speichern

Ergebnisse speichern Sie mit `to_csv` (oder `to_excel`). `index=False` verhindert, dass die Zeilennummern als zusätzliche Spalte gespeichert werden.

In [ ]:
df.to_csv("patienten_bearbeitet.csv", index=False)

# Für deutsches Excel:
# df.to_csv("patienten_excel.csv", index=False, sep=";", decimal=",")

---
## 📝 Übungsaufgaben

### Aufgabe 1: Pulsdruck
Der **Pulsdruck** ist die Differenz zwischen systolischem und diastolischem Blutdruck.
a) Fügen Sie dem DataFrame eine Spalte `pulsdruck` hinzu.

b) Wie groß ist der mittlere Pulsdruck bei Patient\*innen unter 50 und ab 50 Jahren?

In [ ]:
# Ihre Lösung hier

<details>
<summary><b>Lösung anzeigen</b></summary>

```python
df["pulsdruck"] = df["sys_rr"] - df["dia_rr"]

print("unter 50:", df[df["alter"] < 50]["pulsdruck"].mean().round(1))
print("ab 50:   ", df[df["alter"] >= 50]["pulsdruck"].mean().round(1))
```
Alternative mit `groupby`:
```python
df.groupby(df["alter"] >= 50)["pulsdruck"].mean()
```
</details>

### Aufgabe 2: Risikopatient\*innen
Finden Sie alle Patient\*innen, die **gleichzeitig** rauchen, einen Diabetes haben **und** eine Hypertonie. Wie viele sind es? Wie alt sind sie im Mittel?

In [ ]:
# Ihre Lösung hier

<details>
<summary><b>Lösung anzeigen</b></summary>

```python
risiko = df[df["raucher"] & df["diabetes"] & df["hypertonie"]]
print("Anzahl:", len(risiko))
print("Mittleres Alter:", risiko["alter"].mean().round(1))
```
Da `raucher`, `diabetes` und `hypertonie` bereits True/False-Spalten sind, brauchen Sie hier keinen Vergleich wie `== True`.
</details>

### Aufgabe 3: Altersgruppen
a) Teilen Sie die Patient\*innen mit `pd.cut` in die Altersgruppen 18–39, 40–59, 60–79 und 80+ ein (neue Spalte `altersgruppe`).

b) Berechnen Sie für jede Altersgruppe: Anzahl Patient\*innen, mittleres Cholesterin und Anteil Hypertonie.

In [ ]:
# Ihre Lösung hier

<details>
<summary><b>Lösung anzeigen</b></summary>

```python
df["altersgruppe"] = pd.cut(df["alter"], bins=[18, 40, 60, 80, 120],
                            labels=["18–39", "40–59", "60–79", "80+"], right=False)

df.groupby("altersgruppe", observed=True).agg(
    anzahl=("patient_id", "count"),
    cholesterin=("cholesterin", "mean"),
    anteil_hypertonie=("hypertonie", "mean"),
).round(2)
```
Die Schreibweise `neuer_name=("spalte", "funktion")` innerhalb von `.agg()` heißt *named aggregation* und sorgt für schöne Spaltennamen.
</details>

---
## 📌 Zusammenfassung

| Was | Code |
|---|---|
| Import | `import pandas as pd` |
| Laden / Speichern | `pd.read_csv("x.csv")`, `df.to_csv("x.csv", index=False)` |
| Überblick | `df.head()`, `df.shape`, `df.info()`, `df.describe()`, `df["x"].value_counts()` |
| Spalten | `df["x"]`, `df[["x", "y"]]` |
| Filtern | `df[df["alter"] > 65]`, `df[(bed1) & (bed2)]` |
| Gezielt | `df.loc[bedingung, ["x", "y"]]`, `df.iloc[0:5, 0:3]` |
| Neue Spalte | `df["bmi"] = df["gewicht"] / (df["groesse"]/100)**2` |
| Kategorisieren | `pd.cut(df["x"], bins=[...], labels=[...])` |
| Fehlende Werte | `df.isna().sum()`, `df.dropna()`, `df["x"].fillna(wert)` |
| Gruppieren | `df.groupby("gruppe")["x"].mean()`, `.agg([...])` |
| Kreuztabelle | `pd.crosstab(df["a"], df["b"])` |
| Sortieren | `df.sort_values("x", ascending=False)` |

**Weiter geht’s mit Notebook 03 – Matplotlib**, wo wir diesen Datensatz visualisieren.